In [ ]:
!which python

In [3]:
%load_ext autoreload
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell

In [4]:
# basic packages
import os
import re
import sys
import datetime
from typing import List, Dict, Tuple, Optional, Any
from itertools import combinations, product
from pathlib import Path
import glob
#import yaml
import tqdm
import multiprocessing as mp

In [5]:
# data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [6]:
# bioinformatics
import pandas as pd
from Bio.Seq import MutableSeq
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from bintools.utils.utils import get_yaml_config

In [7]:
ROOT_dir = Path(os.path.abspath(os.path.join(Path("../")))).__str__()
if ROOT_dir not in sys.path:
    sys.path.append(ROOT_dir)

In [8]:
list_of_geneID_simu: List[str] = get_yaml_config(ROOT_dir+"/configs/configs-CpG.yaml")["simulation"]["geneID"]
list_of_geneID_emp: List[str] = get_yaml_config(ROOT_dir+"/configs/configs-CpG.yaml")["empirical"]["geneID"]

In [30]:
def sign_95(x,):
     return np.sum(x >= 0.95) / x.shape[0] * 100
    
def sign_99(x,):
     return np.sum(x >= 0.99) / x.shape[0] * 100

def sign_90(x,):
     return np.sum(x >= 0.90) / x.shape[0] * 100
     
def prop(x):
     return np.sum(x) / x.shape[0]

def tran(x):
    if x <= 1:
        return 0
    else:
        return 1


def concat(input_dir:str, pattern:str):
     files: List[str] = glob.glob(input_dir + pattern)
     assert len(files) > 0
     list_of_df : List[pd.DataFrame] = []
     for f in files:
          cur_df: pd.DataFrame = pd.read_csv(f,sep="\t")
          list_of_df += [cur_df]
     return pd.concat(list_of_df,axis=0,ignore_index=True)

def recover_data_emp(input_dir, pattern)-> List[pd.DataFrame]:
        
    set_completed: set = set()
    list_of_df: List[pd.DataFrame] = []
    list_of_files: List[str] = glob.glob(input_dir + pattern)
    for f in list_of_files:
        GENEID = f.split("/")[-1].split("-")[0]
        df: pd.DataFrame = pd.read_csv(f, sep="\t")
        df["geneID"] = [GENEID]*df.shape[0]
        if df.shape[0] == 1000:
            list_of_df += [df]
            set_completed.add(GENEID)
        else:
            print(f"Error in {GENEID}")
    print( set(list_of_geneID_emp) - set_completed)
    return pd.concat(list_of_df,ignore_index=True)


def recover_data_sim(input_dir, pattern, expected_n_lines)-> List[pd.DataFrame]:
    list_of_df: List[pd.DataFrame] = []
    list_of_files: List[str] = glob.glob(input_dir + pattern)
    if len(list_of_files) == 0:
        print(f"Error: No files found in {input_dir} with pattern {pattern}")
        return False
    
    print(f"Found {len(list_of_files)} files in {input_dir} with pattern {pattern}")
    for f in list_of_files:
        GENEID = f.split("/")[-1].split("-")[0]
        OMEGA = float(f.split("/")[-1].split("-")[1])
        CPG = float(f.split("/")[-1].split("-")[2])
        TBL = float(f.split("/")[-1].split("-")[3])
        DRAWID = int(f.split("/")[-1].split("-")[4])
        df: pd.DataFrame = pd.read_csv(f, sep="\t")
        df["geneID"] = [GENEID]*df.shape[0]
        df["omega"] = [OMEGA]*df.shape[0]
        df["CpG"] = [CPG]*df.shape[0]
        df["tbl"] = [TBL]*df.shape[0]
        df["drawID"] = [DRAWID]*df.shape[0]
        if df.shape[0] == expected_n_lines:
            list_of_df += [df]
    return pd.concat(list_of_df,ignore_index=True)

## Mappings


### Empirical


#### MG-F1x4W

In [ ]:
input_dir = f"{ROOT_dir}/outputs/empirical/pbmpi/MG-F1x4W/"
pattern = "*-A.TsCpGRate"
df_concat_m0gtr = recover_data_emp(input_dir=input_dir,pattern=pattern)

In [ ]:
set(list_of_geneID_emp) - set(df_concat_m0gtr["geneID"].unique())  #list_of_geneID_emp

In [11]:
assert 1000 * 137 == df_concat_m0gtr.shape[0]

In [ ]:
df_concat_m0gtr.groupby(["geneID","mcmcID","type"]).agg(["count"])

In [ ]:
df_concat_m0gtr["CpGRate"] = (df_concat_m0gtr["CG>TG"]+df_concat_m0gtr["CG>CA"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatesyn"] = (df_concat_m0gtr["CG>TGsyn"]+df_concat_m0gtr["CG>CAsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatenonsyn"] = (df_concat_m0gtr["CG>TGnonsyn"]+df_concat_m0gtr["CG>CAnonsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRate12"] = (df_concat_m0gtr["CG>TG12"]+df_concat_m0gtr["CG>CA12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRate23"] = (df_concat_m0gtr["CG>TG23"]+df_concat_m0gtr["CG>CA23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRate31"] = (df_concat_m0gtr["CG>TG31"]+df_concat_m0gtr["CG>CA31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatesyn12"] = (df_concat_m0gtr["CG>TGsyn12"]+df_concat_m0gtr["CG>CAsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatesyn23"] = (df_concat_m0gtr["CG>TGsyn23"]+df_concat_m0gtr["CG>CAsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatesyn31"] = (df_concat_m0gtr["CG>TGsyn31"]+df_concat_m0gtr["CG>CAsyn31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatenonsyn12"] = (df_concat_m0gtr["CG>TGnonsyn12"]+df_concat_m0gtr["CG>CAnonsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatenonsyn23"] = (df_concat_m0gtr["CG>TGnonsyn23"]+df_concat_m0gtr["CG>CAnonsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatenonsyn31"] = (df_concat_m0gtr["CG>TGnonsyn31"]+df_concat_m0gtr["CG>CAnonsyn31"])/df_concat_m0gtr["CG31"]


df_concat_m0gtr.groupby(["geneID","type"]).agg([np.mean,np.std])\
    [[
        "CpGRate","CG>TG","CG>CA","CG",\
        "CpGRatesyn","CG>TGsyn","CG>CAsyn",\
        "CpGRatenonsyn","CG>TGnonsyn","CG>CAnonsyn",\

        "CpGRate12","CG>TG12", "CG>CA12", "CG12",\
        "CpGRate23","CG>TG23", "CG>CA23", "CG23",\
        "CpGRate31","CG>TG31", "CG>CA31", "CG31",\
    
        "CpGRatesyn12","CG>TGsyn12", "CG>CAsyn12", \
        "CpGRatesyn23","CG>TGsyn23", "CG>CAsyn23", \
        "CpGRatesyn31","CG>TGsyn31", "CG>CAsyn31", \
    
        "CpGRatenonsyn12","CG>TGnonsyn12", "CG>CAnonsyn12", \
        "CpGRatenonsyn23","CG>TGnonsyn23", "CG>CAnonsyn23", \
        "CpGRatenonsyn31","CG>TGnonsyn31", "CG>CAnonsyn31", \
        
    ]].round(2)

In [ ]:
df_concat_m0gtr.groupby(["geneID","type"]).agg([np.mean,])[[
        "CpGRate","CG>TG","CG>CA","CG",\
        "CpGRatesyn","CG>TGsyn","CG>CAsyn",\
        "CpGRatenonsyn","CG>TGnonsyn","CG>CAnonsyn",\

        "CpGRate12","CG>TG12", "CG>CA12", "CG12",\
        "CpGRate23","CG>TG23", "CG>CA23", "CG23",\
        "CpGRate31","CG>TG31", "CG>CA31", "CG31",\
    
        "CpGRatesyn12","CG>TGsyn12", "CG>CAsyn12", \
        "CpGRatesyn23","CG>TGsyn23", "CG>CAsyn23", \
        "CpGRatesyn31","CG>TGsyn31", "CG>CAsyn31", \
    
        "CpGRatenonsyn12","CG>TGnonsyn12", "CG>CAnonsyn12", \
        "CpGRatenonsyn23","CG>TGnonsyn23", "CG>CAnonsyn23", \
        "CpGRatenonsyn31","CG>TGnonsyn31", "CG>CAnonsyn31", \
        
    ]].round(2)

In [15]:
dict_of_stats = {}
k = 0
rowiter = iter(df_concat_m0gtr.iterrows())
while ((row_post := next(rowiter, None)) is not None):
    row_pred = next(rowiter)
    dict_of_stats[k] = {
        "CpGRate_post": row_post[1]["CpGRate"],
        "CpGRate_pred": row_pred[1]["CpGRate"],
        "CpGRate_comp": row_post[1]["CpGRate"]>row_pred[1]["CpGRate"],

        "CpGRatesyn_post": row_post[1]["CpGRatesyn"],
        "CpGRatesyn_pred": row_pred[1]["CpGRatesyn"],
        "CpGRatesyn_comp": row_post[1]["CpGRatesyn"]>row_pred[1]["CpGRatesyn"],

        "CpGRatenonsyn_post": row_post[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_pred": row_pred[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_comp": row_post[1]["CpGRatenonsyn"]>row_pred[1]["CpGRatenonsyn"],

        "CpGRate12_post": row_post[1]["CpGRate12"],
        "CpGRate12_pred": row_pred[1]["CpGRate12"],
        "CpGRate12_comp": row_post[1]["CpGRate12"]>row_pred[1]["CpGRate12"],
        "CpGRate23_post": row_post[1]["CpGRate23"],
        "CpGRate23_pred": row_pred[1]["CpGRate23"],
        "CpGRate23_comp": row_post[1]["CpGRate23"]>row_pred[1]["CpGRate23"],
        "CpGRate31_post": row_post[1]["CpGRate31"],
        "CpGRate31_pred": row_pred[1]["CpGRate31"],
        "CpGRate31_comp": row_post[1]["CpGRate31"]>row_pred[1]["CpGRate31"],

        "CpGRatesyn12_post": row_post[1]["CpGRatesyn12"],
        "CpGRatesyn12_pred": row_pred[1]["CpGRatesyn12"],
        "CpGRatesyn12_comp": row_post[1]["CpGRatesyn12"]>row_pred[1]["CpGRatesyn12"],
        "CpGRatesyn23_post": row_post[1]["CpGRatesyn23"],
        "CpGRatesyn23_pred": row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn23_comp": row_post[1]["CpGRatesyn23"]>row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn31_post": row_post[1]["CpGRatesyn31"],
        "CpGRatesyn31_pred": row_pred[1]["CpGRatesyn31"],
        "CpGRatesyn31_comp": row_post[1]["CpGRatesyn31"]>row_pred[1]["CpGRatesyn31"],


        "CpGRatenonsyn12_post": row_post[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_pred": row_pred[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_comp": row_post[1]["CpGRatenonsyn12"]>row_pred[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn23_post": row_post[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_pred": row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_comp": row_post[1]["CpGRatenonsyn23"]>row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn31_post": row_post[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_pred": row_pred[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_comp": row_post[1]["CpGRatenonsyn31"]>row_pred[1]["CpGRatenonsyn31"],
        "mcmcID": row_post[1]["mcmcID"],
        "geneID": row_post[1]["geneID"],
    }
    k+=1

In [ ]:
df_TsCpGRate = pd.DataFrame.from_dict(data=dict_of_stats,orient="index")
df_comp = df_TsCpGRate.groupby(["geneID"]) [[
        "CpGRate_comp",
        "CpGRatesyn_comp",
        "CpGRatenonsyn_comp", 
        "CpGRate12_comp", 
        "CpGRate23_comp", 
        "CpGRate31_comp",

        "CpGRatesyn12_comp", 
        "CpGRatesyn23_comp", 
        "CpGRatesyn31_comp",

        "CpGRatenonsyn12_comp", 
        "CpGRatenonsyn23_comp", 
        "CpGRatenonsyn31_comp",

    ]]\
    .agg([np.mean]).droplevel(level=1,axis=1).reset_index()

In [ ]:
df_comp

In [18]:
df_comp.sort_values(by=["geneID"]).to_csv(f"{ROOT_dir}/reports/map_test_MG-F1x4W.csv",sep="\t")

In [19]:
df_stat = df_comp[[
    "CpGRate_comp",
    "CpGRatesyn_comp",
    "CpGRatenonsyn_comp", 
    
    "CpGRate12_comp", 
    "CpGRate23_comp", 
    "CpGRate31_comp",

    "CpGRatesyn12_comp", 
    "CpGRatesyn23_comp", 
    "CpGRatesyn31_comp",

    "CpGRatenonsyn12_comp", 
    "CpGRatenonsyn23_comp", 
    "CpGRatenonsyn31_comp",
    
    ]].agg([sign_99, sign_95, sign_90 ,"count"])
    
df_stat.round(2).to_csv(ROOT_dir + "/reports/map_test_MG-F1x4W_sign.csv",sep="\t")

In [ ]:
df_stat.round(0).T

### Simulation


#### MG-F1x4W

In [ ]:
input_dir = f"{ROOT_dir}/outputs/simulation/MG-F1x4W/pbmpi/"
pattern = "*-A.TsCpGRate"
df_concat_m0gtr = recover_data_sim(input_dir=input_dir,pattern=pattern, expected_n_lines=1000)


In [ ]:
df_concat_m0gtr.columns

In [ ]:
df_concat_m0gtr["CpGRate"] = (df_concat_m0gtr["CG>TG"]+df_concat_m0gtr["CG>CA"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatesyn"] = (df_concat_m0gtr["CG>TGsyn"]+df_concat_m0gtr["CG>CAsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatenonsyn"] = (df_concat_m0gtr["CG>TGnonsyn"]+df_concat_m0gtr["CG>Cnonsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRate12"] = (df_concat_m0gtr["CG>TG12"]+df_concat_m0gtr["CG>CA12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRate23"] = (df_concat_m0gtr["CG>TG23"]+df_concat_m0gtr["CG>CA23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRate31"] = (df_concat_m0gtr["CG>TG31"]+df_concat_m0gtr["CG>CA31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatesyn12"] = (df_concat_m0gtr["CG>TGsyn12"]+df_concat_m0gtr["CG>CAsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatesyn23"] = (df_concat_m0gtr["CG>TGsyn23"]+df_concat_m0gtr["CG>CAsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatesyn31"] = (df_concat_m0gtr["CG>TGsyn31"]+df_concat_m0gtr["CG>CAsyn31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatenonsyn12"] = (df_concat_m0gtr["CG>TGnonsyn12"]+df_concat_m0gtr["CG>CAnonsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatenonsyn23"] = (df_concat_m0gtr["CG>TGnonsyn23"]+df_concat_m0gtr["CG>CAnonsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatenonsyn31"] = (df_concat_m0gtr["CG>TGnonsyn31"]+df_concat_m0gtr["CG>CAnonsyn31"])/df_concat_m0gtr["CG31"]


df_concat_m0gtr.groupby(["geneID","drawID","omega","CpG","tbl","type"]).agg([np.mean,np.std])\
    [[
        "CpGRate","CG>TG","CG>CA","CG",\
        "CpGRatesyn","CG>TGsyn","CG>CAsyn",\
        "CpGRatenonsyn","CG>TGnonsyn","CG>Cnonsyn",\

        "CpGRate12","CG>TG12", "CG>CA12", "CG12",\
        "CpGRate23","CG>TG23", "CG>CA23", "CG23",\
        "CpGRate31","CG>TG31", "CG>CA31", "CG31",\
    
        "CpGRatesyn12","CG>TGsyn12", "CG>CAsyn12", \
        "CpGRatesyn23","CG>TGsyn23", "CG>CAsyn23", \
        "CpGRatesyn31","CG>TGsyn31", "CG>CAsyn31", \
    
        "CpGRatenonsyn12","CG>TGnonsyn12", "CG>CAnonsyn12", \
        "CpGRatenonsyn23","CG>TGnonsyn23", "CG>CAnonsyn23", \
        "CpGRatenonsyn31","CG>TGnonsyn31", "CG>CAnonsyn31", \
        
    ]].round(2)

In [35]:
dict_of_stats = {}
k = 0
rowiter = iter(df_concat_m0gtr.iterrows())
while ((row_post := next(rowiter, None)) is not None):
    try:
        row_pred = next(rowiter)
    except Exception as e:
        print(e,row_post[0])
    dict_of_stats[k] = {
        "CpGRate_post": row_post[1]["CpGRate"],
        "CpGRate_pred": row_pred[1]["CpGRate"],
        "CpGRate_comp": row_post[1]["CpGRate"]>row_pred[1]["CpGRate"],

        "CpGRatesyn_post": row_post[1]["CpGRatesyn"],
        "CpGRatesyn_pred": row_pred[1]["CpGRatesyn"],
        "CpGRatesyn_comp": row_post[1]["CpGRatesyn"]>row_pred[1]["CpGRatesyn"],

        "CpGRatenonsyn_post": row_post[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_pred": row_pred[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_comp": row_post[1]["CpGRatenonsyn"]>row_pred[1]["CpGRatenonsyn"],

        "CpGRate12_post": row_post[1]["CpGRate12"],
        "CpGRate12_pred": row_pred[1]["CpGRate12"],
        "CpGRate12_comp": row_post[1]["CpGRate12"]>row_pred[1]["CpGRate23"],
        "CpGRate23_post": row_post[1]["CpGRate23"],
        "CpGRate23_pred": row_pred[1]["CpGRate23"],
        "CpGRate23_comp": row_post[1]["CpGRate23"]>row_pred[1]["CpGRate23"],
        "CpGRate31_post": row_post[1]["CpGRate31"],
        "CpGRate31_pred": row_pred[1]["CpGRate31"],
        "CpGRate31_comp": row_post[1]["CpGRate31"]>row_pred[1]["CpGRate31"],

        "CpGRatesyn12_post": row_post[1]["CpGRatesyn12"],
        "CpGRatesyn12_pred": row_pred[1]["CpGRatesyn12"],
        "CpGRatesyn12_comp": row_post[1]["CpGRatesyn12"]>row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn23_post": row_post[1]["CpGRatesyn23"],
        "CpGRatesyn23_pred": row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn23_comp": row_post[1]["CpGRatesyn23"]>row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn31_post": row_post[1]["CpGRatesyn31"],
        "CpGRatesyn31_pred": row_pred[1]["CpGRatesyn31"],
        "CpGRatesyn31_comp": row_post[1]["CpGRatesyn31"]>row_pred[1]["CpGRatesyn31"],


        "CpGRatenonsyn12_post": row_post[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_pred": row_pred[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_comp": row_post[1]["CpGRatenonsyn12"]>row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_post": row_post[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_pred": row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_comp": row_post[1]["CpGRatenonsyn23"]>row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn31_post": row_post[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_pred": row_pred[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_comp": row_post[1]["CpGRatenonsyn31"]>row_pred[1]["CpGRatenonsyn31"],

        "mcmcID": row_post[1]["mcmcID"],
        "geneID": row_post[1]["geneID"],
        "omega" : row_post[1]["omega"],
        "CpG" : row_post[1]["CpG"],
        "tbl" : row_post[1]["tbl"],
        "drawID": row_post[1]["drawID"],
    }
    k+=1

In [ ]:
df_comp = pd.DataFrame.from_dict(data=dict_of_stats,orient="index")\
    .groupby(["geneID","mcmcID","drawID","omega","CpG","tbl"])\
    [[
        "CpGRate_comp",
        "CpGRatesyn_comp",
        "CpGRatenonsyn_comp", 
        "CpGRate12_comp", 
        "CpGRate23_comp", 
        "CpGRate31_comp",

        "CpGRatesyn12_comp", 
        "CpGRatesyn23_comp", 
        "CpGRatesyn31_comp",

        "CpGRatenonsyn12_comp", 
        "CpGRatenonsyn23_comp", 
        "CpGRatenonsyn31_comp",

    ]]\
        .agg([np.mean]).droplevel(level=1,axis=1).reset_index()

In [ ]:
df_comp

In [ ]:
df_comp.sort_values(by=["geneID","omega","CpG","tbl"]).to_csv(ROOT_dir + "/reports/map_test_m0gtr_m0gtr.csv",sep="\t")

In [ ]:
df_comp.groupby(by=["CpG","omega","tbl"])[[
    "CpGRate_comp",
    "CpGRatesyn_comp",
    "CpGRatenonsyn_comp", 
    
    "CpGRate12_comp", 
    "CpGRate23_comp", 
    "CpGRate31_comp",

    "CpGRatesyn12_comp", 
    "CpGRatesyn23_comp", 
    "CpGRatesyn31_comp",

    "CpGRatenonsyn12_comp", 
    "CpGRatenonsyn23_comp", 
    "CpGRatenonsyn31_comp",
    
    ]].agg([sign_99, sign_95, sign_90 ,"count"])